In [ ]:
import numpy as np
import scipy.signal
import tensorstore as ts


## Load trigger data

In [ ]:

def load_stimuli_and_ephys(file_handle, num_channels=10):
  try:
    data = np.fromfile(file_handle, dtype=np.float32)
  except io.UnsupportedOperation:
    data = np.frombuffer(file_handle.read(), dtype=np.float32)
  if data.size % num_channels:
    raise ValueError(f'Data does not fit in num_channels: {num_channels}')
  return data.reshape((-1, num_channels)).T

with open('./stimuli_and_ephys.10chFlt', 'rb') as f:
  stimuli_and_ephys = load_stimuli_and_ephys(f)

In [ ]:
ttls = stimuli_and_ephys[2]
ttls_high = scipy.signal.find_peaks(ttls, distance=500, height=3.55)[0]
ttls_low = scipy.signal.find_peaks(ttls, distance=50, height=1)[0]


In [ ]:

# remove volume imaging start steps and only keep plane imaging steps
low_peaks = np.array([l for l in ttls_low if l not in ttls_high])
high_peaks = ttls_high


In [ ]:
low_peaks

In [ ]:
# Create handle to the remote dataset.
ds = ts.open({
    'open': True,
    # Datasets are generally stored in zarr v3 format ('zarr3').
    # There are a few exceptions, where v2 is used ('zarr').
    'driver': 'zarr3',
    # Path of the dataset we want to load.
    'kvstore': 'gs://zapbench-release/volumes/20240930/segmentation'
}).result()

# Display info about the dataset.
print(ds.schema)
segmentation = np.array(ds)

Functional activity volume. 

Whole-brain activity during fictive behavior was imaged at 406 nm×406 nm×4 μm×914 ms resolution in XYZT, 

yielding a volume of size 2048×1328×72×7879 voxels.

In [ ]:

xi, yi, zi = np.where(segmentation > 0)
# get to zero based indexing fast
cell_id_flat = segmentation[xi, yi, zi].astype(np.uint64)
cell_id_flat -= 1

cell_id_uniq, px_per_cell = np.unique(cell_id_flat, return_counts=True)
num_cells = cell_id_uniq.shape[0]
# cell_id_uniq = 1, 2, ..., N
zs_order = np.lexsort((zi, cell_id_flat))
xi = xi[zs_order]
yi = yi[zs_order]
zi = zi[zs_order]
# cell_id_flat is already sorted


In [ ]:
# Let's make a uint64 key of cellid, z.
# we should assert that cell id's are themselves sorted first.
sort_key = np.left_shift(cell_id_flat, 32) + zi
z_uniq, first_index = np.unique(sort_key, return_index=True)
# this would likely be a no op, maybe assert
cell_id_uniq[:] = cell_id_flat[first_index]
breaks = np.zeros(num_cells, dtype=np.int64)
breaks[:-1] = first_index
px_per_cell_per_z = np.diff(breaks)


In [ ]:
# Create handle to the remote dataset.
ds = ts.open({
    'open': True,
    # Datasets are generally stored in zarr v3 format ('zarr3').
    # There are a few exceptions, where v2 is used ('zarr').
    'driver': 'zarr3',
    # Path of the dataset we want to load.
    'kvstore': 'gs://zapbench-release/volumes/20240930/df_over_f_xyz_chunked/s0'
}).result()

# Display info about the dataset.
print(ds.schema)
volume = ds[:512, :512, :8, 0]
